# create_agent vs a Hand-Built LangGraph: Who Controls the Workflow

`create_agent` is convenient, and it genuinely is a LangGraph graph running underneath. That does not mean it hands you the same control as building the graph yourself. This notebook makes that concrete instead of just asserting it.

Questions this answers:
- what does `create_agent`'s compiled graph actually look like, no matter how many tools you give it
- where can you actually customize its behaviour, and where can't you
- when we call a `create_agent` workflow "sequential" or "parallel", what is really deciding that
- what changes when you build the same workflow as an explicit LangGraph graph instead

Read the function calling notebook first if you have not already, this one assumes you know what a tool call and a `ToolMessage` are.


## Setup

What: install the libraries we need.

Why: worth confirming these import cleanly on whatever version Colab hands you before relying on them.


In [ ]:
!pip install -q -U langgraph langchain langchain-google-genai


## API key

What: load the Gemini key into the environment.

Why: `ChatGoogleGenerativeAI` reads it from the environment rather than us passing it around as a plain string.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    import getpass
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

print("key loaded")


## Part 1: What create_agent actually builds

What: build an agent with a handful of tools and print its compiled graph.

Why: rather than take "it's a graph underneath" on faith, look at it directly.


In [ ]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent


@tool
def lookup_customer_id(name: str) -> str:
    """Look up a customer's account id from their full name."""
    return "CUST-4471"


@tool
def fetch_order_history(customer_id: str) -> str:
    """Fetch recent order history for a given customer id."""
    return "3 orders in the last 90 days, most recent total: $240.00"


@tool
def calculate_discount(customer_id: str, order_total: float) -> str:
    """Calculate a loyalty discount for a customer given an order total."""
    return f"Discount for {customer_id}: ${round(order_total * 0.1, 2)}"


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake_data = {"Tokyo": "22C, clear", "London": "14C, rain", "New York": "18C, cloudy"}
    return fake_data.get(city, "no data for this city")


model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

sequential_agent = create_agent(
    model,
    tools=[lookup_customer_id, fetch_order_history, calculate_discount],
)

print(sequential_agent.get_graph().draw_ascii())


Two real nodes, always: `model` and `tools`. Add a hundred more tools and this shape does not change, it is still the same loop - call the model, if it asked for a tool run it, go back to the model, repeat until it stops asking. All of the "intelligence" about which tool to call, in what order, or how many at once, lives entirely inside the `model` node's decision. The graph itself makes exactly one choice: did the model ask for a tool, yes or no.


## Part 2: Where the flexibility actually lives

What `create_agent` lets you configure, without changing that graph shape:
- `system_prompt` - instructions that steer the model's decisions
- `middleware` - hooks that run before or after the model call, or around a tool call (this is how the HITL notebook's `interrupt()`-inside-a-tool pattern plugs in)
- `response_format` - force a structured final answer instead of free text
- `state_schema` - track extra fields alongside the message list

What it does not let you configure: the shape itself. There is no argument for "always call tool A, then unconditionally run B and C together, then D" - the only routing decision the compiled graph makes is the tool-call check above. Anything more structured than that needs a graph you build yourself, which is the rest of this notebook.


## Part 3: Sequential workflow, two ways

**Through create_agent - order comes from prompting.** Below, the three tools happen to get called in the right order only because the prompt spells out the steps and the model follows along. Nothing in the architecture stops the model from skipping a step or reordering them. Worth trying live: weaken the instructions and see if the order still holds.


In [ ]:
result = sequential_agent.invoke({"messages": [
    ("user", "Find the customer id for Priya Shah, then fetch her order history, "
             "then calculate her loyalty discount on a $240 order, in that order.")
]})

for m in result["messages"]:
    role = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(role, "-> called:", [c["name"] for c in m.tool_calls])
    else:
        print(role, "->", getattr(m, "content", "")[:120])


**As a hand-built graph - order comes from the edges.** This version does not ask a model to decide the order at all. `step_1` always runs before `step_2`, which always runs before `step_3`, guaranteed by the graph, not by anyone's wording of a prompt. Kept deliberately simple below (ids are hardcoded rather than threaded through state) so the ordering point stays clear, a real version would pass each step's output into the next through the state instead.


In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import AIMessage


def step_1(state):
    result = lookup_customer_id.invoke({"name": "Priya Shah"})
    return {"messages": [AIMessage(content=f"customer id: {result}")]}


def step_2(state):
    result = fetch_order_history.invoke({"customer_id": "CUST-4471"})
    return {"messages": [AIMessage(content=f"order history: {result}")]}


def step_3(state):
    result = calculate_discount.invoke({"customer_id": "CUST-4471", "order_total": 240.0})
    return {"messages": [AIMessage(content=f"discount: {result}")]}


builder = StateGraph(MessagesState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)
fixed_sequence_graph = builder.compile()

result = fixed_sequence_graph.invoke({"messages": []})
for m in result["messages"]:
    print(m.content)


## Part 4: Parallel workflow, two ways

**Through create_agent - concurrency comes from the model batching calls.** Same weather-comparison prompt as the function calling notebook. If Gemini decides to ask for all three cities in one turn, the tool-execution node runs them concurrently for you automatically. If it decides to ask one at a time instead, that is three round trips, and there is no setting to force either behaviour.


In [ ]:
weather_agent = create_agent(model, tools=[get_weather])

result = weather_agent.invoke({"messages": [
    ("user", "Compare the weather in Tokyo, London, and New York.")
]})

for m in result["messages"]:
    if getattr(m, "tool_calls", None):
        print("model asked for", len(m.tool_calls), "call(s) at once:", [c["args"] for c in m.tool_calls])


**As a hand-built graph - concurrency is structural.** No model in this graph at all, three branches run because the edges say so, in the same execution step, every single time. This is the pattern to reach for when "these things must run together" is a requirement rather than something you are hoping the model decides.


In [ ]:
import operator
from typing import Annotated, TypedDict


class ParallelState(TypedDict):
    results: Annotated[list, operator.add]


def check_tokyo(state):
    return {"results": [get_weather.invoke({"city": "Tokyo"})]}


def check_london(state):
    return {"results": [get_weather.invoke({"city": "London"})]}


def check_new_york(state):
    return {"results": [get_weather.invoke({"city": "New York"})]}


def summarize(state):
    return {"results": [f"collected {len(state['results'])} reports"]}


builder = StateGraph(ParallelState)
builder.add_node("check_tokyo", check_tokyo)
builder.add_node("check_london", check_london)
builder.add_node("check_new_york", check_new_york)
builder.add_node("summarize", summarize)
builder.add_edge(START, "check_tokyo")
builder.add_edge(START, "check_london")
builder.add_edge(START, "check_new_york")
builder.add_edge("check_tokyo", "summarize")
builder.add_edge("check_london", "summarize")
builder.add_edge("check_new_york", "summarize")
builder.add_edge("summarize", END)
structural_parallel_graph = builder.compile()

print(structural_parallel_graph.get_graph().draw_ascii())
print()
print(structural_parallel_graph.invoke({"results": []}))


Compare this shape to Part 1's two-node loop. The branches here run in parallel because of the edges pointing out of `START`, not because anything decided to batch them, `operator.add` on the `results` field is what lets three branches write into the same list without overwriting each other.


## Recap: which one do you actually want

**create_agent**
- a few lines to a working agent
- workflow shape is fixed: model, tools, loop
- sequencing and parallelism are emergent, a consequence of the model's own reasoning, steered but not guaranteed by your prompt and tool design
- extend behaviour through `system_prompt`, `middleware`, `response_format`, not by reshaping the graph

**Hand-built LangGraph**
- more code, you are writing the routing yourself
- workflow shape is whatever you draw: fixed sequences, guaranteed parallel branches, steps that don't involve an LLM at all if you don't want them to
- sequencing and parallelism are structural, true regardless of what any model decides
- the right call once a workflow has a hard requirement: a step that must never be skipped, a fixed order for compliance reasons, or genuinely guaranteed concurrency

A reasonable default for the room: start with `create_agent` for anything exploratory, and drop down to a hand-built graph the moment a step in the workflow cannot be left to the model's discretion.
